In [0]:
CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

In [0]:
from pyspark.sql.functions import (col, regexp_replace, to_timestamp, trim,
                                   when, dayofweek, hour, date_format, to_date,
                                   date_trunc, split, size, lit)

bt = spark.table("bronze.transactions")
mcc = spark.table("bronze.mcc_codes")
fraud = spark.table("bronze.fraud_labels")

silver_tx = (bt
    .select(
        col("id").cast("int").alias("transaction_id"),
        to_timestamp("date", "yyyy-MM-dd HH:mm:ss").alias("transaction_ts"),
        col("client_id").cast("int").alias("client_id"),
        col("card_id").cast("int").alias("card_id"),
        regexp_replace(col("amount"), r"[$,]", "").cast("decimal(12,2)").alias("amount"),
        trim(col("use_chip")).alias("use_chip"),
        col("merchant_id").cast("int").alias("merchant_id"),
        trim(col("merchant_city")).alias("merchant_city"),
        trim(col("merchant_state")).alias("merchant_state"),
        col("zip").cast("double").cast("int").alias("zip"),
        col("mcc").cast("int").alias("mcc"),
        when(trim(col("errors")) == "", None).otherwise(trim(col("errors"))).alias("errors"),
    )
    # derived columns the gold layer needs — compute once here, not 14 times downstream
    .withColumn("transaction_date", to_date("transaction_ts"))
    .withColumn("day_of_week", date_format("transaction_ts", "EEEE"))
    .withColumn("day_of_week_num", dayofweek("transaction_ts"))
    .withColumn("transaction_hour", hour("transaction_ts"))
    .withColumn("time_of_day",
        when(col("transaction_hour").between(6, 11), "Morning")
        .when(col("transaction_hour").between(12, 17), "Afternoon")
        .when(col("transaction_hour").between(18, 21), "Evening")
        .otherwise("Night"))
    .withColumn("year_month", date_format("transaction_ts", "yyyy-MM"))
    .withColumn("week_start", date_trunc("week", col("transaction_ts")).cast("date"))
    .withColumn("has_error", col("errors").isNotNull())
    .withColumn("error_count",
        when(col("errors").isNull(), 0).otherwise(size(split(col("errors"), ","))))
)

# mcc description
silver_tx = (silver_tx.alias("t")
    .join(mcc.select(col("mcc").cast("int").alias("mcc"), "mcc_description").alias("m"),
          on="mcc", how="left"))

# fraud label — LEFT join, see note below
silver_tx = (silver_tx.alias("t")
    .join(fraud.select(col("transaction_id").cast("int").alias("transaction_id"),
                       col("is_fraud").alias("is_fraud_raw")).alias("f"),
          on="transaction_id", how="left")
    .withColumn("is_fraud", when(col("is_fraud_raw") == "Yes", True)
                            .when(col("is_fraud_raw") == "No", False)
                            .otherwise(None))
    .withColumn("is_labeled", col("is_fraud_raw").isNotNull())
    .drop("is_fraud_raw"))

(silver_tx.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year_month")
    .saveAsTable("silver.transactions"))

In [0]:
bc = spark.table("bronze.cards")

silver_cards = (bc.select(
        col("id").cast("int").alias("card_id"),
        col("client_id").cast("int").alias("client_id"),
        trim(col("card_brand")).alias("card_brand"),
        trim(col("card_type")).alias("card_type"),
        col("card_number").alias("card_number"),
        to_date(col("expires"), "MM/yyyy").alias("expires_date"),
        col("cvv").alias("cvv"),
        (trim(col("has_chip")) == "YES").alias("has_chip"),
        col("num_cards_issued").cast("int").alias("num_cards_issued"),
        regexp_replace(col("credit_limit"), r"[$,]", "").cast("decimal(12,2)").alias("credit_limit"),
        to_date(col("acct_open_date"), "MM/yyyy").alias("acct_open_date"),
        col("year_pin_last_changed").cast("int").alias("year_pin_last_changed"),
        (trim(col("card_on_dark_web")) == "Yes").alias("card_on_dark_web"),
    ))

silver_cards.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.cards")

In [0]:
bu = spark.table("bronze.users")

silver_users = (bu.select(
        col("id").cast("int").alias("client_id"),
        col("current_age").cast("int").alias("current_age"),
        col("retirement_age").cast("int").alias("retirement_age"),
        col("birth_year").cast("int").alias("birth_year"),
        col("birth_month").cast("int").alias("birth_month"),
        trim(col("gender")).alias("gender"),
        trim(col("address")).alias("address"),
        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude"),
        regexp_replace(col("per_capita_income"), r"[$,]", "").cast("decimal(12,2)").alias("per_capita_income"),
        regexp_replace(col("yearly_income"), r"[$,]", "").cast("decimal(12,2)").alias("yearly_income"),
        regexp_replace(col("total_debt"), r"[$,]", "").cast("decimal(12,2)").alias("total_debt"),
        col("credit_score").cast("int").alias("credit_score"),
        col("num_credit_cards").cast("int").alias("num_credit_cards"),
    ))

silver_users.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.users")

In [0]:
from pyspark.sql.functions import count, sum as _sum, min as _min, max as _max

tx = spark.table("silver.transactions")

print("rows:", tx.count())                                    # 13305915
print("labeled:", tx.filter("is_labeled").count())            # ~8.9M
print("fraud:", tx.filter("is_fraud").count())
print("null amount:", tx.filter(col("amount").isNull()).count())        # 0
print("null ts:", tx.filter(col("transaction_ts").isNull()).count())    # 0
print("null mcc_desc:", tx.filter(col("mcc_description").isNull()).count())
display(tx.select(_min("transaction_ts"), _max("transaction_ts")))

rows: 13305915
labeled: 8914963
fraud: 13332
null amount: 0
null ts: 0
null mcc_desc: 0


min(transaction_ts),max(transaction_ts)
2010-01-01T00:01:00.000Z,2019-10-31T23:59:00.000Z
